# Fabric Defect Detection — Google Colab

**Hướng dẫn:**
1. Runtime → Change runtime type → **T4 GPU**
2. Chạy từng cell theo thứ tự
3. Cell 9 tạo link Gradio public để demo

**Lần đầu:** upload `code.zip` + `data_tsfabric_T1.zip` lên Drive (chạy `scripts/pack_for_colab.py` trên máy local trước)

## Cell 1 — Kiểm tra GPU

In [ ]:
import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
import psutil
print(f'RAM : {psutil.virtual_memory().total / 1e9:.1f} GB')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/defect_detection'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted → {DRIVE_DIR}')
print('Files:', os.listdir(DRIVE_DIR))

## Cell 3 — Giải nén code + data

In [ ]:
import os, zipfile

PROJECT_DIR = '/content/defect_detection'
CODE_ZIP    = f'{DRIVE_DIR}/code.zip'
DATA_ZIP    = f'{DRIVE_DIR}/data_tsfabric_T1.zip'

# Giải nén code
if not os.path.exists(f'{PROJECT_DIR}/configs'):
    print('Giải nén code...')
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        z.extractall('/content')
    print('✓ Code OK')
else:
    print('✓ Code đã có')

# Tự detect prefix trong zip để extract đúng chỗ
# zip cũ: defect_detection/data/... → extract vào /content/
# zip mới: data/...                 → extract vào PROJECT_DIR
data_dir = f'{PROJECT_DIR}/data/pass/train/tsfabric_T1'
if not os.path.exists(data_dir) or len(os.listdir(data_dir)) == 0:
    print('Giải nén data (~310 MB)...')
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        first = z.namelist()[0]
        extract_to = '/content' if first.startswith('defect_detection/') else PROJECT_DIR
        print(f'  Zip prefix: {first[:40]} → extract to {extract_to}')
        z.extractall(extract_to)
    print('✓ Data OK')
else:
    print('✓ Data đã có')

os.chdir(PROJECT_DIR)
print(f'pass images : {len(os.listdir("data/pass/train/tsfabric_T1"))}')
print(f'fail images : {len(os.listdir("data/fail/train/tsfabric_T1"))}')
print(f'masks       : {len(os.listdir("data/fail/masks/tsfabric_T1"))}')

## Cell 4 — Cài dependencies

In [ ]:
import sys, os
sys.path.insert(0, '/content/defect_detection')
os.chdir('/content/defect_detection')

# SAM2 từ source (bắt buộc)
if not os.path.exists('/content/segment-anything-2'):
    !git clone -q https://github.com/facebookresearch/segment-anything-2 /content/segment-anything-2
    !pip install -e /content/segment-anything-2 -q
    print('✓ SAM2 installed')
else:
    print('✓ SAM2 already installed')

# Pin versions đã kiểm tra hoạt động trên Colab T4
!pip install -q \
    "transformers==4.44.0" \
    "peft>=0.12.0" \
    "torchao>=0.16.0" \
    gradio \
    faiss-cpu \
    rich \
    accelerate \
    bitsandbytes

print('✓ All packages installed')

## Cell 5 — Download SAM2 weights

In [ ]:
import os
os.chdir('/content/defect_detection')
os.makedirs('weights/sam2', exist_ok=True)

sam2_path = 'weights/sam2/sam2_hiera_large.pt'
if not os.path.exists(sam2_path):
    print('Downloading SAM2 weights (~900 MB)...')
    !wget -q --show-progress \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt \
        -O {sam2_path}
    print('✓ SAM2 weights downloaded')
else:
    print('✓ SAM2 weights already exist')

## Cell 6 — Copy checkpoints từ Drive

In [ ]:
import shutil, os
os.chdir('/content/defect_detection')
os.makedirs('outputs/checkpoints', exist_ok=True)

CKPT_DRIVE = f'{DRIVE_DIR}/checkpoints'
if os.path.exists(CKPT_DRIVE):
    for f in os.listdir(CKPT_DRIVE):
        src = f'{CKPT_DRIVE}/{f}'
        dst = f'outputs/checkpoints/{f}'
        if not os.path.exists(dst):
            if os.path.isdir(src): shutil.copytree(src, dst)
            else: shutil.copy2(src, dst)
            print(f'  Copied: {f}')
    print('✓ Checkpoints loaded')
else:
    print('Chưa có checkpoint trên Drive — cần train (Cell 7)')

!ls outputs/checkpoints/

## Cell 7 — Train

Bỏ qua nếu Cell 6 đã load đủ 3 checkpoint.

> **Lưu ý Colab RAM:** Stage 1 dùng config nhẹ hơn (`batch=2`, `max_samples=5000`) để tránh OOM.
> Stage 1 mất ~15–20 phút, Stage 2 mất ~2–3 giờ trên T4.

In [ ]:
import os, sys
os.chdir('/content/defect_detection')
sys.path.insert(0, '/content/defect_detection')
os.environ['PYTHONPATH'] = '/content/defect_detection'

ckpt = 'outputs/checkpoints'
stage1_ok = os.path.exists(f'{ckpt}/stage1_tsfabric_T1_bank.npy')
stage2_ok = os.path.exists(f'{ckpt}/stage2_tsfabric_T1_lora.pt')
stage3_ok = os.path.exists(f'{ckpt}/stage3_tsfabric_T1_lora')
print(f'Stage 1: {"✓" if stage1_ok else "✗"}')
print(f'Stage 2: {"✓" if stage2_ok else "✗"}')
print(f'Stage 3: {"✓" if stage3_ok else "✗"}')

In [ ]:
# === Train Stage 1 (bỏ qua nếu đã có checkpoint) ===
import os
os.chdir('/content/defect_detection')

if not os.path.exists('outputs/checkpoints/stage1_tsfabric_T1_bank.npy'):
    # Giảm batch + coreset để tránh OOM trên Colab
    !sed -i 's/"batch_size": 4/"batch_size": 2/' configs/stage1_config.py
    !sed -i 's/"max_samples": 10000/"max_samples": 5000/' configs/stage1_config.py
    !sed -i 's/GREEDY_LIMIT = 50_000/GREEDY_LIMIT = 20_000/' stage1_anomaly/memory_bank.py
    print('Stage 1 config patched (batch=2, max_samples=5000, greedy_limit=20K)')
    print('Training Stage 1 (~15-20 phút)...')
    !python -m stage1_anomaly.train --category tsfabric_T1
else:
    print('✓ Stage 1 đã có checkpoint')

In [ ]:
# === Train Stage 2 (bỏ qua nếu đã có checkpoint) ===
import os
os.chdir('/content/defect_detection')

if not os.path.exists('outputs/checkpoints/stage1_tsfabric_T1_bank.npy'):
    print('❌ Stage 1 chưa có checkpoint. Chạy cell Stage 1 trước.')
elif not os.path.exists('outputs/checkpoints/stage2_tsfabric_T1_lora.pt'):
    print('Training Stage 2 (~2-3 giờ trên T4)...')
    !python -m stage2_seg.train --category tsfabric_T1 --eval
else:
    print('✓ Stage 2 đã có checkpoint')

In [ ]:
# === Train Stage 3 (bỏ qua nếu đã có checkpoint hoặc không có vlm_ann) ===
import os
os.chdir('/content/defect_detection')

vlm_count = len(os.listdir('data/vlm_ann')) if os.path.exists('data/vlm_ann') else 0
print(f'VLM annotations: {vlm_count} files')

if not os.path.exists('outputs/checkpoints/stage3_tsfabric_T1_lora'):
    if vlm_count < 50:
        print('⚠ Không đủ annotation (cần ≥50) — Stage 3 dùng zero-shot')
    else:
        print('Training Stage 3...')
        !python -m stage3_vlm.lora_train --category tsfabric_T1
else:
    print('✓ Stage 3 đã có checkpoint')

## Cell 8 — Lưu checkpoint về Drive

In [ ]:
import shutil, os
os.chdir('/content/defect_detection')

CKPT_DRIVE = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DRIVE, exist_ok=True)

for f in os.listdir('outputs/checkpoints'):
    src = f'outputs/checkpoints/{f}'
    dst = f'{CKPT_DRIVE}/{f}'
    if not os.path.exists(dst):
        if os.path.isdir(src): shutil.copytree(src, dst)
        else: shutil.copy2(src, dst)
        print(f'  Saved: {f}')

print(f'✓ Checkpoints saved → {CKPT_DRIVE}')
!ls {CKPT_DRIVE}

## Cell 9 — Chạy Gradio UI
Copy link `https://xxxx.gradio.live` để demo.

In [ ]:
import sys, os
sys.path.insert(0, '/content/defect_detection')
os.chdir('/content/defect_detection')
os.environ['PYTHONPATH'] = '/content/defect_detection'

!python app.py --share --category tsfabric_T1

---
## (Optional) Test 1 ảnh không cần UI

In [ ]:
import sys, os
sys.path.insert(0, '/content/defect_detection')
os.chdir('/content/defect_detection')

TEST_IMAGE = 'data/fail/train/tsfabric_T1/000003.jpeg'
!python inference/run_pipeline.py --image {TEST_IMAGE} --category tsfabric_T1